## 📦 1. Kurulum

In [ ]:
# Unsloth + wandb (Colab'da bir kez çalıştır, sonra runtime restart gerekebilir)
# NOT: trl/transformers/peft/accelerate ayrıca --upgrade edilmiyor.
# Unsloth kendi bağımlılıklarını pinliyor; ayrı upgrade sürüm drift'ine yol açar.
%%capture
!pip install -U unsloth wandb

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## 🔧 2. Konfigürasyon

In [ ]:
# ────────────────────────────────────────────────
#  Tüm ayarlar buradan yönetilir
# ────────────────────────────────────────────────

# Model
MODEL_NAME       = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"  # 4-bit quantized, direkt hazır
MAX_SEQ_LENGTH   = 2048   # T4 kullanıyorsan 1024 yap
DTYPE            = None   # None → otomatik (bf16 A100, fp16 T4)
LOAD_IN_4BIT     = True

# LoRA
LORA_R           = 16
LORA_ALPHA       = 16
LORA_DROPOUT     = 0.0    # Unsloth optimized: 0 daha iyi
TARGET_MODULES   = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"]

# Eğitim
OUTPUT_DIR       = "./mentorx-qwen25-7b-lora"
NUM_EPOCHS       = 3
BATCH_SIZE       = 4      # T4 kullanıyorsan 2 yap
GRAD_ACCUM       = 4      # Effective batch = BATCH_SIZE * GRAD_ACCUM = 16
LEARNING_RATE    = 2e-4
WARMUP_STEPS     = 50
LR_SCHEDULER     = "cosine"
WEIGHT_DECAY     = 0.01
MAX_GRAD_NORM    = 1.0
LOGGING_STEPS    = 10
SAVE_STEPS       = 100
EVAL_STEPS       = 100
SEED             = 42

# Veri yolları (Google Drive mount edildiyse güncelle)
TRAIN_FILE       = "train_v2.jsonl"   # Colab'a yüklendiğinde
VAL_FILE         = "val_v2.jsonl"

# HuggingFace Hub (opsiyonel - kaydet/paylaş)
HF_PUSH          = False
HF_REPO          = "kullanici-adi/mentorx-qwen25-7b"  # HF username/repo

# WandB (opsiyonel)
USE_WANDB        = False
WANDB_PROJECT    = "mentorx-sft"

print("✅ Konfigürasyon yüklendi")
print(f"   Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

✅ Konfigürasyon yüklendi
   Effective batch size: 16


## 📂 3. Veri Yükleme

In [ ]:
# Seçenek A: Manuel upload (basit)
from google.colab import files

print("train_v2.jsonl ve val_v2.jsonl dosyalarını seç:")
uploaded = files.upload()

for fname in uploaded:
    print(f"✅ Yüklendi: {fname} ({len(uploaded[fname])/1024:.1f} KB)")

train_v2.jsonl ve val_v2.jsonl dosyalarını seç:


Saving train_v2.jsonl to train_v2.jsonl
Saving val_v2.jsonl to val_v2.jsonl
✅ Yüklendi: train_v2.jsonl (8091.7 KB)
✅ Yüklendi: val_v2.jsonl (886.2 KB)


In [ ]:
# Seçenek B: Google Drive'dan yükle (önerilir - daha hızlı)
# Eğer Drive kullanıyorsan bu hücreyi çalıştır, üstteki hücreyi atla

# from google.colab import drive
# drive.mount('/content/drive')

# TRAIN_FILE = "/content/drive/MyDrive/mentorx/train_v2.jsonl"
# VAL_FILE   = "/content/drive/MyDrive/mentorx/val_v2.jsonl"
# OUTPUT_DIR = "/content/drive/MyDrive/mentorx/mentorx-qwen25-7b-lora"

print("Drive hücreleri yorum satırında — gerekirse aç")

Drive hücreleri yorum satırında — gerekirse aç


In [ ]:
from datasets import load_dataset
import json

# JSONL → HuggingFace Dataset
dataset = load_dataset(
    "json",
    data_files={"train": TRAIN_FILE, "validation": VAL_FILE}
)

print(f"Train örnekleri : {len(dataset['train'])}")
print(f"Val örnekleri   : {len(dataset['validation'])}")

# Örnek görüntüle
sample = dataset["train"][0]
print(f"\n--- Örnek diyalog ({len(sample['messages'])} mesaj) ---")
for msg in sample["messages"][:3]:  # ilk 3 mesajı göster
    role = msg["role"].upper()
    preview = msg["content"][:80] + "..." if len(msg["content"]) > 80 else msg["content"]
    print(f"[{role}]: {preview}")

# Şema doğrulama
def validate_split(ds, name):
    for i, row in enumerate(ds):
        msgs = row.get("messages")
        assert isinstance(msgs, list) and len(msgs) >= 3, f"{name}[{i}] invalid messages"
        assert msgs[0].get("role") == "system",    f"{name}[{i}] first role must be system"
        assert msgs[-1].get("role") == "assistant", f"{name}[{i}] last role must be assistant"
        for j, m in enumerate(msgs):
            assert m.get("role") in {"system", "user", "assistant"}, f"{name}[{i}] bad role @{j}"
            c = m.get("content")
            assert isinstance(c, str) and c.strip(), f"{name}[{i}] empty content @{j}"

validate_split(dataset["train"],      "train")
validate_split(dataset["validation"], "validation")
print("✅ Dataset şeması doğrulandı")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Train örnekleri : 1824
Val örnekleri   : 201

--- Örnek diyalog (13 mesaj) ---
[SYSTEM]: Sen MentorX adlı bir Türkçe Otomata Teorisi tutorusun. Öğrenciye asla doğrudan c...
[USER]: Hocam, `b ∪ Σ*a` dilini tanıyan bir NFA taslağı düşündüm. Başlangıçtan iki kola ...
[ASSISTANT]: Güzel bir başlangıç taslağı. Peki, bu taslağınızın 'ab' dizisini kabul etmemesi ...
✅ Dataset şeması doğrulandı


## 🤖 4. Model & Tokenizer Yükleme (Unsloth)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = DTYPE,
    load_in_4bit  = LOAD_IN_4BIT,
)

print(f"✅ Model yüklendi: {MODEL_NAME}")
print(f"   Parametre sayısı: {model.num_parameters()/1e9:.2f}B")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Model yüklendi: unsloth/Qwen2.5-7B-Instruct-bnb-4bit
   Parametre sayısı: 7.62B


## 🔗 5. LoRA Adapter Ekleme

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                   = LORA_R,
    target_modules      = TARGET_MODULES,
    lora_alpha          = LORA_ALPHA,
    lora_dropout        = LORA_DROPOUT,
    bias                = "none",
    use_gradient_checkpointing = "unsloth",  # Unsloth'un kendi optimize versiyonu
    random_state        = SEED,
    use_rslora          = False,
    loftq_config        = None,
)

# Eğitilebilir parametre sayısını göster
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA eklendi")
print(f"   Eğitilebilir: {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.2f}%)")

Unsloth 2026.5.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ LoRA eklendi
   Eğitilebilir: 40.4M / 4.39B (0.92%)


## 🗂️ 6. Veri Formatı — Chat Template Uygulama

In [ ]:
from unsloth.chat_templates import get_chat_template

# Qwen2.5 chat template uygula
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def format_conversations(examples):
    """
    Her örnekteki messages listesini Qwen2.5 chat template'iyle
    tek bir string'e dönüştürür.
    Loss masking ayrı hücrede (train_on_responses_only) uygulanır.
    """
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize          = False,
            add_generation_prompt = False,
        )
        texts.append(text)
    return {"text": texts}

# Dataset'e uygula
dataset_formatted = dataset.map(
    format_conversations,
    batched = True,
    remove_columns = dataset["train"].column_names,
)

print(f"✅ Chat template uygulandı")
print(f"\n--- Formatlanmış örnek (ilk 500 karakter) ---")
print(dataset_formatted["train"][0]["text"][:500])
print("...")

Map:   0%|          | 0/1824 [00:00<?, ? examples/s]

Map:   0%|          | 0/201 [00:00<?, ? examples/s]

✅ Chat template uygulandı

--- Formatlanmış örnek (ilk 500 karakter) ---
<|im_start|>system
Sen MentorX adlı bir Türkçe Otomata Teorisi tutorusun. Öğrenciye asla doğrudan cevap vermezsin, her mesajında yönlendirici sorular sorarsın, max 3 cümleyle konuşursun.<|im_end|>
<|im_start|>user
Hocam, `b ∪ Σ*a` dilini tanıyan bir NFA taslağı düşündüm. Başlangıçtan iki kola ayrılıyorum: bir kol sadece 'b' ile nihai bir duruma gider. Diğer kol ise `Σ*` için kendi üzerinde döngülerle ('a' ve 'b' ile) ve ardından bir 'a' ile nihai bir duruma ulaşır.<|im_end|>
<|im_start|>assistan
...


In [ ]:
# Token uzunluklarını kontrol et — MAX_SEQ_LENGTH aşılıyor mu?
sample_texts = dataset_formatted["train"]["text"]  # tüm train seti
lengths = [len(tokenizer.encode(t)) for t in sample_texts]

import numpy as np
print(f"Token uzunlukları (ilk 200 örnek):")
print(f"  Min    : {min(lengths)}")
print(f"  Max    : {max(lengths)}")
print(f"  Ort    : {np.mean(lengths):.0f}")
print(f"  P95    : {np.percentile(lengths, 95):.0f}")
print(f"  MAX_SEQ: {MAX_SEQ_LENGTH}")

truncated = sum(1 for l in lengths if l > MAX_SEQ_LENGTH)
print(f"  Kesilen: {truncated} / {len(lengths)} örnek ({100*truncated/len(lengths):.1f}%)")

Token uzunlukları (ilk 200 örnek):
  Min    : 410
  Max    : 5640
  Ort    : 1406
  P95    : 3461
  MAX_SEQ: 2048
  Kesilen: 269 / 1824 örnek (14.7%)


## 🏋️ 7. Eğitim

## 🎯 7. Sadece Assistant Yanıtlarında Loss Hesapla

SFTTrainer varsayılan olarak tüm token'larda (system + user + assistant) loss hesaplar.
Biz sadece **assistant yanıtlarını** öğretmek istiyoruz — `train_on_responses_only` user/system token'larını maskeler.

In [ ]:
# WandB (opsiyonel — kullanmak istersen)
import os

if USE_WANDB:
    import wandb
    wandb.login()  # API key sorar
    os.environ["WANDB_PROJECT"] = WANDB_PROJECT
    report_to = "wandb"
    print(f"✅ WandB aktif: {WANDB_PROJECT}")
else:
    os.environ["WANDB_DISABLED"] = "true"
    report_to = "none"
    print("WandB devre dışı")

WandB devre dışı


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq, set_seed
from unsloth import is_bfloat16_supported
import math

# Deterministik seed
set_seed(SEED)

# Patch 8: eval/save adımlarını epoch büyüklüğüne göre dinamik hesapla
steps_per_epoch = max(1, math.ceil(len(dataset_formatted["train"]) / (BATCH_SIZE * GRAD_ACCUM)))
eval_save_steps = max(20, steps_per_epoch // 2)
print(f"Epoch başına adım : {steps_per_epoch}")
print(f"Eval/save her     : {eval_save_steps} adımda bir")

# ────────────────────────────────────────────────
#  Verimlilik Seçeneği:
#
#  USE_PACKING = False  →  Dynamic Padding (önerilen)
#    Batch içindeki en uzun örneğe göre padding yapar.
#    train_on_responses_only ile güvenli çalışır.
#
#  USE_PACKING = True   →  Sequence Packing (daha agresif)
#    Kısa diyalogları birleştirip tek sequence gibi gösterir.
#    Daha hızlı ama train_on_responses_only maskelemesi
#    karışabilir — dikkatli kullan.
# ────────────────────────────────────────────────

USE_PACKING         = False  # True → Seçenek B

# Dynamic Padding collator
data_collator = None
if not USE_PACKING:
    data_collator = DataCollatorForSeq2Seq(
        tokenizer          = tokenizer,
        model              = model,
        padding            = "longest",  # Sabit max değil, batch'teki en uzuna göre
        pad_to_multiple_of = 8,          # Tensor Core optimizasyonu
    )
    print("✅ Dynamic Padding aktif")
else:
    print("⚠️  Packing aktif — train_on_responses_only maskelemesini kontrol et!")

trainer = SFTTrainer(
    model                         = model,
    tokenizer                     = tokenizer,
    train_dataset                 = dataset_formatted["train"],
    eval_dataset                  = dataset_formatted["validation"],
    dataset_text_field            = "text",
    max_seq_length                = MAX_SEQ_LENGTH,
    dataset_num_proc              = 2,
    packing                       = USE_PACKING,
    data_collator                 = data_collator,
    args = TrainingArguments(
        output_dir                = OUTPUT_DIR,
        num_train_epochs          = NUM_EPOCHS,
        per_device_train_batch_size  = BATCH_SIZE,
        per_device_eval_batch_size   = 1,
        gradient_accumulation_steps  = GRAD_ACCUM,
        warmup_steps              = WARMUP_STEPS,
        learning_rate             = LEARNING_RATE,
        lr_scheduler_type         = LR_SCHEDULER,
        weight_decay              = WEIGHT_DECAY,
        max_grad_norm             = MAX_GRAD_NORM,
        fp16                      = not is_bfloat16_supported(),
        bf16                      = is_bfloat16_supported(),
        logging_steps             = LOGGING_STEPS,
        save_steps                = eval_save_steps,
        eval_strategy             = "steps",
        eval_steps                = eval_save_steps,
        load_best_model_at_end    = True,
        metric_for_best_model     = "eval_loss",
        greater_is_better         = False,
        save_total_limit          = 3,
        seed                      = SEED,
        report_to                 = report_to,
        optim                     = "adamw_8bit",
        dataloader_pin_memory     = True,
    ),
)

print(f"✅ Trainer hazır")
print(f"   Train          : {len(dataset_formatted['train'])} örnek")
print(f"   Val            : {len(dataset_formatted['validation'])} örnek")
print(f"   load_best_model: eval_loss (adım 100, 200, 300)")

steps_per_epoch = len(dataset_formatted["train"]) // (BATCH_SIZE * GRAD_ACCUM)
total_steps     = steps_per_epoch * NUM_EPOCHS
print(f"   Epoch başına adım (max): ~{steps_per_epoch}")
print(f"   Toplam adım (max)      : ~{total_steps}")

Epoch başına adım : 114
Eval/save her     : 57 adımda bir
✅ Dynamic Padding aktif


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1824 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/201 [00:00<?, ? examples/s]

✅ Trainer hazır
   Train          : 1824 örnek
   Val            : 201 örnek
   load_best_model: eval_loss (adım 100, 200, 300)
   Epoch başına adım (max): ~114
   Toplam adım (max)      : ~342


In [ ]:
from unsloth.chat_templates import train_on_responses_only

# Qwen2.5 token sınırlarını tanıt — loss sadece assistant yanıtlarında hesaplanacak
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

# Maskelemenin doğru çalıştığını doğrula
# -100 olan token'lar loss hesabına katılmaz (system + user maskelendi)
sample = trainer.train_dataset[0]
input_ids = sample["input_ids"]
labels    = sample["labels"]

masked   = sum(1 for l in labels if l == -100)
unmasked = sum(1 for l in labels if l != -100)
total    = len(labels)

print(f"✅ train_on_responses_only aktif")
print(f"   Toplam token   : {total}")
print(f"   Maskelenen     : {masked}  (system + user → loss yok)")
print(f"   Loss hesaplanan: {unmasked} (sadece assistant yanıtları)")
print(f"   Oran           : %{100*unmasked/total:.1f} assistant token")

assert unmasked > 0, "Assistant mask başarısız: hiç loss token yok."
assert masked   > 0, "Mask başarısız: user/system maskelenmemiş görünüyor."
print("✅ Maskeleme doğrulandı")

Map (num_proc=16):   0%|          | 0/1824 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/1824 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/201 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/201 [00:00<?, ? examples/s]

✅ train_on_responses_only aktif
   Toplam token   : 1024
   Maskelenen     : 693  (system + user → loss yok)
   Loss hesaplanan: 331 (sadece assistant yanıtları)
   Oran           : %32.3 assistant token
✅ Maskeleme doğrulandı


In [ ]:
# Eğitim öncesi VRAM durumu
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"VRAM toplam : {max_memory:.1f} GB")
print(f"VRAM kullanım (şu an): {start_gpu_memory:.1f} GB")

GPU: NVIDIA A100-SXM4-40GB
VRAM toplam : 39.5 GB
VRAM kullanım (şu an): 5.4 GB


In [ ]:
# 🚀 Eğitimi Başlat
# train_on_responses_only uygulandığını doğrula
s = trainer.train_dataset[0]
assert "labels" in s and any(x != -100 for x in s["labels"]), \
    "train_on_responses_only uygulanmamış — eğitimi durdur!"

print("Eğitim başlıyor...")
trainer_stats = trainer.train()

# Eğitim sonrası istatistikler
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\n✅ Eğitim tamamlandı!")
print(f"   Süre           : {trainer_stats.metrics['train_runtime']/60:.1f} dakika")
print(f"   Train loss     : {trainer_stats.metrics['train_loss']:.4f}")
print(f"   Peak VRAM      : {used_memory:.1f} GB / {max_memory:.1f} GB")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Eğitim başlıyor...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,824 | Num Epochs = 3 | Total steps = 342
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
57,1.335140,1.281028
114,1.208873,1.170462
171,1.066225,1.138441
228,1.042076,1.106364
285,0.883710,1.120683
342,0.871120,1.116655


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


✅ Eğitim tamamlandı!
   Süre           : 18.9 dakika
   Train loss     : 1.1071
   Peak VRAM      : 25.0 GB / 39.5 GB


## 💾 8. Model Kaydetme

In [ ]:
# LoRA adaptörlerini kaydet (küçük dosya, sadece ince ayar ağırlıkları)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ LoRA adaptörleri kaydedildi: {OUTPUT_DIR}")

# Dosya boyutlarını göster
import os
total_size = 0
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in files:
        fpath = os.path.join(root, f)
        total_size += os.path.getsize(fpath)
print(f"   Toplam boyut: {total_size/1024/1024:.1f} MB")

Unsloth: Restored added_tokens_decoder metadata in ./mentorx-qwen25-7b-lora/tokenizer_config.json.


✅ LoRA adaptörleri kaydedildi: ./mentorx-qwen25-7b-lora
   Toplam boyut: 895.8 MB


In [ ]:
# Opsiyonel: LoRA + base model birleştirilmiş (merged) kayıt
# NOT: Bu ~15GB yer kaplar, Drive'a kaydetmek için Drive mount edin

# MERGED_DIR = "/content/drive/MyDrive/mentorx/mentorx-qwen25-7b-merged"
# model.save_pretrained_merged(
#     MERGED_DIR,
#     tokenizer,
#     save_method = "merged_16bit",  # ya da "merged_4bit" (daha küçük)
# )
# print(f"✅ Birleştirilmiş model kaydedildi: {MERGED_DIR}")

print("Birleştirilmiş model kaydı yorum satırında — gerekirse aç")

Birleştirilmiş model kaydı yorum satırında — gerekirse aç


In [ ]:
# Opsiyonel: GGUF formatı (Ollama / llama.cpp ile kullanım için)

# model.save_pretrained_gguf(
#     "mentorx-qwen25-7b-gguf",
#     tokenizer,
#     quantization_method = "q4_k_m",  # Q4_K_M = iyi denge
# )
# print("✅ GGUF kaydedildi")

print("GGUF kaydı yorum satırında — gerekirse aç")

GGUF kaydı yorum satırında — gerekirse aç


In [ ]:
# LoRA ağırlıklarını ZIP yapıp indir
import shutil
from google.colab import files

zip_path = "mentorx_lora_weights"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
print(f"✅ ZIP oluşturuldu: {zip_path}.zip")

# Dosya boyutunu kontrol et
zip_size = os.path.getsize(f"{zip_path}.zip") / 1024 / 1024
print(f"   Boyut: {zip_size:.1f} MB")

files.download(f"{zip_path}.zip")
print("İndirme başlatıldı...")

✅ ZIP oluşturuldu: mentorx_lora_weights.zip
   Boyut: 787.7 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

İndirme başlatıldı...


## 🧪 9. Çıkarım Testi (Inference)

In [ ]:
# Inference modu — 2x hız (Unsloth optimize)
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584, padding_idx=151665)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [ ]:
# MentorX'i test et
test_messages = [
    {
        "role": "system",
        "content": "Sen MentorX adlı bir Türkçe Otomata Teorisi tutorusun. "
                   "Öğrenciye asla doğrudan cevap vermezsin, her mesajında "
                   "yönlendirici sorular sorarsın, max 3 cümleyle konuşursun."
    },
    {
        "role": "user",
        "content": "Hocam, bir DFA ile NFA arasındaki temel fark nedir?"
    }
]

# Tokenize et
inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize           = True,
    add_generation_prompt = True,
    return_tensors     = "pt"
).to("cuda")

# Üret
with torch.no_grad():
    outputs = model.generate(
        input_ids          = inputs,
        max_new_tokens     = 200,
        temperature        = 0.7,
        top_p              = 0.9,
        repetition_penalty = 1.1,
        do_sample          = True,
        pad_token_id       = tokenizer.eos_token_id,
    )

# Sadece üretilen kısmı decode et
response = tokenizer.decode(
    outputs[0][inputs.shape[-1]:],
    skip_special_tokens = True
)

print("=" * 60)
print("🎓 MentorX Yanıtı:")
print("=" * 60)
print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

🎓 MentorX Yanıtı:
Harika bir başlangıç noktası! Bir DFA'nın tanımını nasıl kısaca özetlersin? Özellikle geçiş fonksiyonu ve durum kümesi hakkında ne düşünüyorsun?


In [ ]:
# İnteraktif çok turlu sohbet testi
def chat_with_mentorx(user_message, history=None):
    """MentorX ile çok turlu konuşma"""
    if history is None:
        history = [{
            "role": "system",
            "content": "Sen MentorX adlı bir Türkçe Otomata Teorisi tutorusun. "
                       "Öğrenciye asla doğrudan cevap vermezsin, her mesajında "
                       "yönlendirici sorular sorarsın, max 3 cümleyle konuşursun."
        }]

    history.append({"role": "user", "content": user_message})

    inputs = tokenizer.apply_chat_template(
        history,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = "pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids          = inputs,
            max_new_tokens     = 200,
            temperature        = 0.7,
            top_p              = 0.9,
            repetition_penalty = 1.1,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens = True
    )

    history.append({"role": "assistant", "content": response})
    return response, history


# Test sohbeti
history = None
test_questions = [
    "Pumping Lemma nedir?",
    "O zaman nasıl kullanmalıyım?",
]

for q in test_questions:
    print(f"\n👤 Öğrenci: {q}")
    response, history = chat_with_mentorx(q, history)
    print(f"🎓 MentorX: {response}")
    print("-" * 50)

## 📝 Notlar

### Hiperparametre önerileri
| Parametre | A100 (40GB) | T4 (16GB) |
|-----------|-------------|----------|
| max_seq_length | 2048 | 1024 |
| batch_size | 4 | 2 |
| grad_accum | 4 | 8 |
| lora_r | 16 | 8 |

### Sonraki adımlar
- **Değerlendirme:** LLM-as-Judge (llm_judge_v3.py) ile fine-tuned modeli test et
- **Karşılaştırma:** Base Qwen2.5-7B vs fine-tuned Sokratik davranış
- **GGUF:** Ollama ile yerel deployment için GGUF export
- **Python CS1:** Aynı pipeline'ı yeni modül için tekrar kullan

### LoRA ağırlıklarını sonradan yüklemek için
```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "./mentorx-qwen25-7b-lora",  # LoRA klasörü
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
```